# Exploring the Notebook
There are many features and functions available in Databricks notebooks.

## Let's start using our catalog


In [0]:
%sql
USE CATALOG demo_workspace;

## Next Let's Create a table for products

In [0]:
%sql
CREATE TABLE product_info (
    product_id INT,
    product_name STRING,
    category STRING,
    price DOUBLE,
    quantity INT
);

## Next let's insert some data

In [0]:
%sql
INSERT INTO product_info (product_id, product_name, category, price, quantity) 
VALUES (1, 'Winter Jacket', 'Clothing', 79.99, 100);

INSERT INTO product_info (product_id, product_name, category, price, quantity)
VALUES (2, 'Microwave', 'Kitchen', 249.95, 30),
       (3, 'Coffee Maker', 'Kitchen', 69.99, 50),
       (4, 'Board Game', 'Toys', 45.99, 30),
       (5, 'Teddy Bear', 'Toys', 19.99, 70),
       (6, 'iPad', 'Electronics', 799.99, 20),
       (7, 'Smartwatch', 'Electronics', 199.99, 40),
       (8, 'Running Shoes', 'Clothing', 89.99, 150);

## OK, we've got some data, let's see what's up

In [0]:
%sql
SELECT * FROM product_info

## We've got data in there, but...
The transactions in the table have been completed via two files that have been added. Look back at the above notebook to see that there are two insert operations. Even though the latter has more records inserted, they each count as a separate transaction and a separate file affiliated. We can see this in act via the DESCRIBE function.

In [0]:
%sql
DESCRIBE DETAIL product_info

Thus, numFiles shows number of files, amongst many other things such as location of the file.

## Let's take a look at the tables file structure

In [0]:
# let's look at table structure
display(spark.sql("DESCRIBE HISTORY product_info"))

In [0]:
# List the actual data files backing the table
files = spark.read.table("product_info").inputFiles()
for f in files:
    print(f)

## Let's change the price of our goods to account for inflation

In [0]:
%sql
UPDATE product_info 
SET price = price * 1.10

In [0]:
# Let's now see that change in the table
display(spark.sql("DESCRIBE DETAIL product_info"))

## Interesting Files are now one
When we execute the change on the entire database, it appends that new updated file and 'discards' any data files that are not valid in the current version. Thus we only have the new file with all the new prices.

In [0]:
display(spark.sql("DESCRIBE DETAIL product_info"))

In [0]:
%sql
DESCRIBE HISTORY product_info

In [0]:
# read the underlying JSON from delta log of product_info
dbutils.fs.head(dbutils.fs.ls("/user/hive/warehouse/product_info/_delta_log/00000000000000000000.json")[0].path)

## Let's Look at Delta Time Travel
Time travel allows you to retrieve previous versions of data in Delta Lake tables.

In [0]:
%sql
DESCRIBE HISTORY product_info

## What if we wanted to query the older version?
Options are using either timestamp or version number

In [0]:
%sql
SELECT * FROM product_info VERSION AS OF 2

## In some cases, you may need to revert back to previous versions
RESTORE TABLE will help you do that.

`RESTORE TABLE <table_name> TO VERSION/TIMESTAMP AS OF <timestamp/version>`

## Let's say we accidentally DELETED! the table

In [0]:
%sql
DELETE FROM product_info

## We'll now need to restore the table

In [0]:
%sql
RESTORE TABLE product_info TO VERSION AS OF 3

## Delta Lake also provide table optimization
This is done through compacting small files into larger ones

In [0]:
%sql
OPTIMIZE product_info

### Furthermore OPTIMIZE can leverage Z-Order
`OPTIMIZE TABLE product_info`

`ZORDER BY <column_names>`

In [0]:
%sql
OPTIMIZE product_info
ZORDER BY product_id

In [0]:
%sql
DESCRIBE DETAIL product_info

In [0]:
%sql
DESCRIBE HISTORY product_info

In [0]:
files = spark.read.table("product_info").inputFiles()
for f in files:
    print(f)

## I need to now make a small insert to demonstrate VACUUM

In [0]:
%sql
INSERT INTO product_info (product_id, product_name, category, price, quantity)
VALUES (9, 'Refridgerator', 'Kitchen', 499.95, 7)

## Let's check the files

In [0]:
files = spark.read.table("product_info").inputFiles()
for f in files:
    print(f)

In [0]:
%sql
OPTIMIZE product_info
ZORDER BY product_id

## VACUUM allows us to clean up unwanted files saving storage space and cost
`VACUUM <table_name> [RETAIN num HOURS]`

Default and minimum retention period is 7 days. DB will prevent you from deleting files that are younger than 7 days. 

Once you VACUUM, you cannot time travel back toa version older than that period. Consider company policy on data retention.

In [0]:
files = spark.read.table("product_info").inputFiles()
for f in files:
    print(f)

In [0]:
%sql
VACUUM product_info RETAIN 0 HOURS

## This will allow us to disable the check and run our VACUUM

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false

## Let's Drop the table now and permanenly erase any data

In [0]:
%sql
DROP TABLE product_info

In [0]:
%sql
SELECT * FROM product_info